
# Otimização do novo controlador fuzzy usando a PIRNN multivariada

Este notebook integra:

1. O controlador fuzzy utilizado no trabalho de João.
2. A PIRNN que estima os incrementos `dX`, `dY` e `dTheta`.
3. A reconstrução da trajetória estimada `x_hat`, `y_hat` e `theta_hat`.
4. A função de custo com os termos:
   
   $$J = w_1E_r+w_2E_{\mathrm{traj}}+w_3E_u+w_4E_a$$
   
5. A otimização dos cinco parâmetros fuzzy por `differential_evolution`.

> **Atenção:** ajuste os caminhos dos arquivos e, principalmente, a função
> `preparar_entrada_pirnn()` conforme a ordem exata das entradas usada no treinamento
> da sua PIRNN.


In [ ]:

# ============================================================
# 1. Bibliotecas e configurações
# ============================================================

import os
import warnings
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import tensorflow as tf

from tensorflow import keras
from scipy.optimize import differential_evolution

import skfuzzy as fuzz
from skfuzzy import control as ctrl

warnings.filterwarnings("ignore")
np.set_printoptions(suppress=True, precision=6)

# ------------------------------------------------------------
# Caminhos: altere conforme sua organização de arquivos
# ------------------------------------------------------------

MODEL_PATH = "best_pirnn_model.keras"

# Scalers utilizados no treinamento da PIRNN
X_SCALER_PATH = "x_scaler.gz"
Y_SCALER_PATH = "y_scaler.gz"

# Arquivo contendo a trajetória de referência
REFERENCE_PATH = "trajetoria_referencia.csv"

# Colunas esperadas no arquivo de referência
XR_COL = "xr"
YR_COL = "yr"

# Se existir referência angular, informe a coluna.
# Caso não exista, será calculada a orientação da trajetória.
THETA_REF_COL = None

# ------------------------------------------------------------
# Constantes do robô
# ------------------------------------------------------------

R_WHEEL = 0.0328       # raio da roda [m]
L_HALF = 0.0615        # metade da distância entre rodas [m]
DT = 0.07              # período de amostragem [s]

# Número de passos temporais usados pela PIRNN
TIMESTEPS = 15

# Tipo de saída da PIRNN:
# "incrementos" = rede retorna dX, dY, dTheta
# "estado"      = rede retorna X, Y, Theta diretamente
PIRNN_OUTPUT_TYPE = "incrementos"

# Os incrementos dX e dY são:
# "global" = referencial global
# "body"   = referencial do robô
INCREMENT_FRAME = "global"

# Entradas usadas no treinamento da PIRNN.
# Ajuste caso seu modelo tenha outra ordem.
PIRNN_INPUT_COLUMNS = ["PwmD", "PwmE", "sPwm", "dPwm"]

# Limites PWM
PWM_MIN = -1.0
PWM_MAX = 1.0

# Pesos iniciais da função de custo.
# Recomenda-se normalizar os termos antes de aplicar os pesos.
WEIGHTS = np.array([0.30, 0.40, 0.10, 0.20], dtype=float)

# Parâmetros fuzzy:
# [z_dp, pff_dp, mp_dp, pff_m, mp_m]
PARAMETER_BOUNDS = [
    (1e-4, np.pi / 4),
    (1e-4, np.pi / 4),
    (1e-4, np.pi / 4),
    (1e-4, np.pi / 2),
    (1e-4, np.pi / 2),
]

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Configuração carregada.")


In [ ]:

# ============================================================
# 2. Carregamento do modelo PIRNN e dos scalers
# ============================================================

def carregar_modelos():
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"Modelo não encontrado: {MODEL_PATH}. "
            "Altere MODEL_PATH para o arquivo correto."
        )

    modelo = keras.models.load_model(MODEL_PATH, compile=False)

    x_scaler = None
    y_scaler = None

    if os.path.exists(X_SCALER_PATH):
        x_scaler = joblib.load(X_SCALER_PATH)
    else:
        print("Aviso: X scaler não encontrado. Será usada entrada sem normalização.")

    if os.path.exists(Y_SCALER_PATH):
        y_scaler = joblib.load(Y_SCALER_PATH)
    else:
        print("Aviso: Y scaler não encontrado. Será usada saída sem desnormalização.")

    print("Modelo:", MODEL_PATH)
    print("Entrada esperada:", modelo.input_shape)
    print("Saída produzida:", modelo.output_shape)

    return modelo, x_scaler, y_scaler


# Execute quando os arquivos estiverem disponíveis:
# modelo_pirnn, x_scaler, y_scaler = carregar_modelos()


In [ ]:

# ============================================================
# 3. Controlador fuzzy de João
# ============================================================

def controlador_fuzzy_vetor(parametros_vetor, phi_input, l_input=1.0):
    """
    Retorna:
        vf: velocidade linear desejada
        wf: velocidade angular desejada

    Parâmetros:
        [z_dp, pff_dp, mp_dp, pff_m, mp_m]
    """

    z_dp, pff_dp, mp_dp, pff_m, mp_m = np.asarray(
        parametros_vetor, dtype=float
    )

    PHI = ctrl.Antecedent(np.linspace(-np.pi, np.pi, 181), "PHI")
    L = ctrl.Antecedent(np.linspace(0, 5, 181), "L")
    VF = ctrl.Consequent(np.linspace(-0.15, 0.15, 181), "VF")
    WF = ctrl.Consequent(np.linspace(-0.5, 0.5, 181), "WF")

    PHI["EN"] = fuzz.gaussmf(PHI.universe, -np.pi, z_dp)
    PHI["PNR"] = fuzz.gaussmf(PHI.universe, pff_m - np.pi, pff_dp)
    PHI["GN"] = fuzz.gaussmf(PHI.universe, mp_m - np.pi, mp_dp)
    PHI["MN"] = fuzz.gaussmf(PHI.universe, -mp_m, mp_dp)
    PHI["PNF"] = fuzz.gaussmf(PHI.universe, -pff_m, pff_dp)
    PHI["Z"] = fuzz.gaussmf(PHI.universe, 0, z_dp)
    PHI["PPF"] = fuzz.gaussmf(PHI.universe, pff_m, pff_dp)
    PHI["MP"] = fuzz.gaussmf(PHI.universe, mp_m, mp_dp)
    PHI["GP"] = fuzz.gaussmf(PHI.universe, np.pi - mp_m, mp_dp)
    PHI["PPR"] = fuzz.gaussmf(PHI.universe, np.pi - pff_m, pff_dp)
    PHI["EP"] = fuzz.gaussmf(PHI.universe, np.pi, z_dp)

    L["MP"] = fuzz.zmf(L.universe, 0.01, 0.02)
    L["G"] = fuzz.smf(L.universe, 0.01, 0.02)

    VF["R"] = fuzz.pimf(VF.universe, -0.151, -0.15, -0.1, -0.05)
    VF["MR"] = fuzz.pimf(VF.universe, -0.1, -0.05, -0.05, 0)
    VF["P"] = fuzz.pimf(VF.universe, -0.05, 0, 0, 0.05)
    VF["MF"] = fuzz.pimf(VF.universe, 0, 0.05, 0.05, 0.1)
    VF["F"] = fuzz.pimf(VF.universe, 0.05, 0.1, 0.15, 0.151)

    WF["H"] = fuzz.trimf(WF.universe, [-0.5, -0.5, 0])
    WF["N"] = fuzz.trimf(WF.universe, [-0.5, 0, 0.5])
    WF["AH"] = fuzz.trimf(WF.universe, [0, 0.5, 0.5])

    rules = [
        ctrl.Rule(L["MP"], (VF["P"], WF["N"])),
        ctrl.Rule(PHI["EN"] & L["G"], (VF["R"], WF["N"])),
        ctrl.Rule(PHI["PNR"] & L["G"], (VF["MR"], WF["AH"])),
        ctrl.Rule(PHI["GN"] & L["G"], (VF["P"], WF["AH"])),
        ctrl.Rule(PHI["MN"] & L["G"], (VF["P"], WF["H"])),
        ctrl.Rule(PHI["PNF"] & L["G"], (VF["MF"], WF["H"])),
        ctrl.Rule(PHI["Z"] & L["G"], (VF["F"], WF["N"])),
        ctrl.Rule(PHI["PPF"] & L["G"], (VF["MF"], WF["AH"])),
        ctrl.Rule(PHI["MP"] & L["G"], (VF["P"], WF["AH"])),
        ctrl.Rule(PHI["GP"] & L["G"], (VF["P"], WF["H"])),
        ctrl.Rule(PHI["PPR"] & L["G"], (VF["MR"], WF["H"])),
        ctrl.Rule(PHI["EP"] & L["G"], (VF["R"], WF["N"])),
    ]

    sistema = ctrl.ControlSystem(rules)
    simulador = ctrl.ControlSystemSimulation(sistema)

    phi_input = float(np.clip(phi_input, -np.pi, np.pi))
    l_input = float(np.clip(l_input, 0, 5))

    simulador.input["PHI"] = phi_input
    simulador.input["L"] = l_input
    simulador.compute()

    return np.array([
        simulador.output["VF"],
        simulador.output["WF"]
    ], dtype=float)


In [ ]:

# ============================================================
# 4. Conversão de velocidade linear/angular para PWM
# ============================================================

def velocidade_para_rodas(vf, wf):
    """
    Conversão cinemática:
        phi_d = (v + L*omega)/R
        phi_e = (v - L*omega)/R
    """
    phi_d = (vf + L_HALF * wf) / R_WHEEL
    phi_e = (vf - L_HALF * wf) / R_WHEEL
    return phi_d, phi_e


def rodas_para_pwm(phi_d, phi_e):
    """
    Conversão provisória de velocidade angular para PWM.

    IMPORTANTE:
    Substitua esta função pela mesma relação identificada no seu
    treinamento experimental, se houver um modelo PWM -> velocidade.

    Aqui é utilizada uma saturação simples para permitir a execução.
    """
    max_phi = 10.0

    pwm_d = np.clip(phi_d / max_phi, PWM_MIN, PWM_MAX)
    pwm_e = np.clip(phi_e / max_phi, PWM_MIN, PWM_MAX)

    return pwm_d, pwm_e


In [ ]:

# ============================================================
# 5. Preparação da entrada da PIRNN
# ============================================================

def preparar_entrada_pirnn(pwm_d, pwm_e, historico_pwm):
    """
    Monta a sequência de entrada da PIRNN.

    A ordem deve ser exatamente a mesma utilizada no treinamento.
    Para:
        ["PwmD", "PwmE", "sPwm", "dPwm"]

    temos:
        sPwm = PwmD + PwmE
        dPwm = PwmD - PwmE
    """

    s_pwm = pwm_d + pwm_e
    d_pwm = pwm_d - pwm_e

    linha = {
        "PwmD": pwm_d,
        "PwmE": pwm_e,
        "sPwm": s_pwm,
        "dPwm": d_pwm,
    }

    vetor = [linha[col] for col in PIRNN_INPUT_COLUMNS]
    historico_pwm.append(vetor)

    sequencia = np.asarray(historico_pwm, dtype=float)

    if x_scaler is not None:
        shape_original = sequencia.shape
        sequencia_2d = sequencia.reshape(-1, shape_original[-1])
        sequencia_2d = x_scaler.transform(sequencia_2d)
        sequencia = sequencia_2d.reshape(shape_original)

    return sequencia.reshape(1, TIMESTEPS, len(PIRNN_INPUT_COLUMNS))


def prever_pirnn(modelo, entrada, y_scaler=None):
    pred = modelo.predict(entrada, verbose=0)
    pred = np.asarray(pred)

    # Aceita saídas [1, 3], [1, 1, 3] ou estruturas semelhantes
    pred = pred.reshape(-1)

    if y_scaler is not None:
        pred = y_scaler.inverse_transform(pred.reshape(1, -1)).reshape(-1)

    return pred


In [ ]:

# ============================================================
# 6. Métricas da função de custo
# ============================================================

def calcular_Er(xr, yr, x_hat, y_hat):
    erro = np.sqrt((xr - x_hat)**2 + (yr - y_hat)**2)
    return float(np.sqrt(np.mean(erro**2)))


def calcular_Etraj(xr, yr, x_hat, y_hat):
    erros = []

    for k in range(1, len(xr)):
        dx = xr[k] - xr[k-1]
        dy = yr[k] - yr[k-1]
        denominador = np.sqrt(dx**2 + dy**2)

        if denominador < 1e-12:
            continue

        numerador = abs(
            dx * (yr[k-1] - y_hat[k])
            - (xr[k-1] - x_hat[k]) * dy
        )

        erros.append(numerador / denominador)

    if not erros:
        return 0.0

    return float(np.sqrt(np.mean(np.asarray(erros)**2)))


def calcular_Eu(phi_d, phi_e):
    return float(np.mean(np.abs(phi_d) + np.abs(phi_e)))


def calcular_Ea(phi_d, phi_e):
    velocidade_linear = R_WHEEL / 2.0 * (phi_d + phi_e)
    aceleracao = np.diff(velocidade_linear) / DT

    if len(aceleracao) == 0:
        return 0.0

    return float(np.sqrt(np.mean(aceleracao**2)))


def calcular_metricas(xr, yr, x_hat, y_hat, phi_d, phi_e):
    return np.array([
        calcular_Er(xr, yr, x_hat, y_hat),
        calcular_Etraj(xr, yr, x_hat, y_hat),
        calcular_Eu(phi_d, phi_e),
        calcular_Ea(phi_d, phi_e),
    ], dtype=float)


In [ ]:

# ============================================================
# 7. Simulação em malha fechada com a PIRNN
# ============================================================

def simular_malha_fechada_pirnn(
    parametros_fuzzy,
    modelo,
    xr,
    yr,
    theta_ref,
    x0=0.0,
    y0=0.0,
    theta0=0.0,
    x_scaler_model=None,
    y_scaler_model=None,
    l_input=1.0,
):
    """
    Simula controlador fuzzy + PIRNN em malha fechada.

    A PIRNN é utilizada como modelo virtual do robô.
    """

    global x_scaler, y_scaler
    x_scaler = x_scaler_model
    y_scaler = y_scaler_model

    n = len(xr)

    x_hat = np.zeros(n)
    y_hat = np.zeros(n)
    theta_hat = np.zeros(n)

    phi_d_hist = np.zeros(n)
    phi_e_hist = np.zeros(n)
    pwm_d_hist = np.zeros(n)
    pwm_e_hist = np.zeros(n)

    x_hat[0] = x0
    y_hat[0] = y0
    theta_hat[0] = theta0

    historico_pwm = deque(maxlen=TIMESTEPS)

    # Inicialização compatível com a janela temporal da PIRNN
    vetor_zero = [0.0] * len(PIRNN_INPUT_COLUMNS)
    for _ in range(TIMESTEPS):
        historico_pwm.append(vetor_zero.copy())

    for k in range(1, n):
        # Erro angular da trajetória
        erro_theta = theta_ref[k-1] - theta_hat[k-1]
        erro_theta = np.arctan2(
            np.sin(erro_theta),
            np.cos(erro_theta)
        )

        # Controlador fuzzy
        vf, wf = controlador_fuzzy_vetor(
            parametros_fuzzy,
            phi_input=erro_theta,
            l_input=l_input
        )

        # Comandos das rodas
        phi_d, phi_e = velocidade_para_rodas(vf, wf)
        pwm_d, pwm_e = rodas_para_pwm(phi_d, phi_e)

        phi_d_hist[k] = phi_d
        phi_e_hist[k] = phi_e
        pwm_d_hist[k] = pwm_d
        pwm_e_hist[k] = pwm_e

        # Entrada da PIRNN
        entrada = preparar_entrada_pirnn(
            pwm_d,
            pwm_e,
            historico_pwm
        )

        pred = prever_pirnn(
            modelo,
            entrada,
            y_scaler=y_scaler_model
        )

        if len(pred) < 3:
            raise ValueError(
                "A PIRNN não retornou três valores. "
                "Verifique se a saída é [dX, dY, dTheta] ou [X, Y, Theta]."
            )

        valor_1, valor_2, valor_3 = pred[:3]

        if PIRNN_OUTPUT_TYPE == "incrementos":
            dx = valor_1
            dy = valor_2
            dtheta = valor_3

            if INCREMENT_FRAME == "body":
                # Conversão de incremento no referencial do robô
                dx_global = (
                    dx * np.cos(theta_hat[k-1])
                    - dy * np.sin(theta_hat[k-1])
                )
                dy_global = (
                    dx * np.sin(theta_hat[k-1])
                    + dy * np.cos(theta_hat[k-1])
                )
                dx, dy = dx_global, dy_global

            x_hat[k] = x_hat[k-1] + dx
            y_hat[k] = y_hat[k-1] + dy
            theta_hat[k] = theta_hat[k-1] + dtheta

        elif PIRNN_OUTPUT_TYPE == "estado":
            x_hat[k] = valor_1
            y_hat[k] = valor_2
            theta_hat[k] = valor_3

        else:
            raise ValueError(
                "PIRNN_OUTPUT_TYPE deve ser 'incrementos' ou 'estado'."
            )

    return {
        "x_hat": x_hat,
        "y_hat": y_hat,
        "theta_hat": theta_hat,
        "phi_d": phi_d_hist,
        "phi_e": phi_e_hist,
        "pwm_d": pwm_d_hist,
        "pwm_e": pwm_e_hist,
    }


In [ ]:

# ============================================================
# 8. Função de custo normalizada
# ============================================================

def normalizar_metricas(metricas, metricas_base):
    denominador = np.maximum(np.abs(metricas_base), 1e-12)
    return metricas / denominador


def funcao_custo(
    parametros_fuzzy,
    modelo,
    xr,
    yr,
    theta_ref,
    x0,
    y0,
    theta0,
    metricas_base,
    pesos,
    x_scaler_model=None,
    y_scaler_model=None,
    l_input=1.0,
    imprimir=False,
):
    try:
        resultado = simular_malha_fechada_pirnn(
            parametros_fuzzy=parametros_fuzzy,
            modelo=modelo,
            xr=xr,
            yr=yr,
            theta_ref=theta_ref,
            x0=x0,
            y0=y0,
            theta0=theta0,
            x_scaler_model=x_scaler_model,
            y_scaler_model=y_scaler_model,
            l_input=l_input,
        )

        metricas = calcular_metricas(
            xr,
            yr,
            resultado["x_hat"],
            resultado["y_hat"],
            resultado["phi_d"],
            resultado["phi_e"],
        )

        metricas_norm = normalizar_metricas(
            metricas,
            metricas_base
        )

        custo = float(np.dot(pesos, metricas_norm))

        if not np.isfinite(custo):
            return 1e12

        if imprimir:
            print("Parâmetros:", np.asarray(parametros_fuzzy))
            print("Métricas:", metricas)
            print("Métricas normalizadas:", metricas_norm)
            print("Custo:", custo)

        return custo

    except Exception as exc:
        print("Falha na avaliação:", repr(exc))
        return 1e12


In [ ]:

# ============================================================
# 9. Leitura da trajetória de referência
# ============================================================

def carregar_referencia(path=REFERENCE_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Arquivo de referência não encontrado: {path}. "
            "Altere REFERENCE_PATH."
        )

    df = pd.read_csv(path)

    xr = df[XR_COL].to_numpy(dtype=float)
    yr = df[YR_COL].to_numpy(dtype=float)

    if THETA_REF_COL is not None and THETA_REF_COL in df.columns:
        theta_ref = df[THETA_REF_COL].to_numpy(dtype=float)
    else:
        dx = np.gradient(xr)
        dy = np.gradient(yr)
        theta_ref = np.unwrap(np.arctan2(dy, dx))

    return df, xr, yr, theta_ref


# Execute quando o arquivo estiver disponível:
# referencia_df, xr, yr, theta_ref = carregar_referencia()


In [ ]:

# ============================================================
# 10. Preparação da referência de normalização
# ============================================================

def obter_metricas_base(
    modelo,
    xr,
    yr,
    theta_ref,
    parametros_base,
    x0,
    y0,
    theta0,
    x_scaler_model=None,
    y_scaler_model=None,
    l_input=1.0,
):
    resultado_base = simular_malha_fechada_pirnn(
        parametros_fuzzy=parametros_base,
        modelo=modelo,
        xr=xr,
        yr=yr,
        theta_ref=theta_ref,
        x0=x0,
        y0=y0,
        theta0=theta0,
        x_scaler_model=x_scaler_model,
        y_scaler_model=y_scaler_model,
        l_input=l_input,
    )

    metricas_base = calcular_metricas(
        xr,
        yr,
        resultado_base["x_hat"],
        resultado_base["y_hat"],
        resultado_base["phi_d"],
        resultado_base["phi_e"],
    )

    return metricas_base, resultado_base


In [ ]:

# ============================================================
# 11. Otimização por Differential Evolution
# ============================================================

def otimizar_controlador(
    modelo,
    xr,
    yr,
    theta_ref,
    parametros_base,
    x0,
    y0,
    theta0,
    x_scaler_model=None,
    y_scaler_model=None,
    l_input=1.0,
    pesos=WEIGHTS,
    maxiter=50,
    popsize=8,
):
    print("Calculando métricas do controlador base...")

    metricas_base, resultado_base = obter_metricas_base(
        modelo=modelo,
        xr=xr,
        yr=yr,
        theta_ref=theta_ref,
        parametros_base=parametros_base,
        x0=x0,
        y0=y0,
        theta0=theta0,
        x_scaler_model=x_scaler_model,
        y_scaler_model=y_scaler_model,
        l_input=l_input,
    )

    print("Métricas base:")
    print("Er    =", metricas_base[0])
    print("Etraj =", metricas_base[1])
    print("Eu    =", metricas_base[2])
    print("Ea    =", metricas_base[3])

    def objetivo(parametros):
        return funcao_custo(
            parametros_fuzzy=parametros,
            modelo=modelo,
            xr=xr,
            yr=yr,
            theta_ref=theta_ref,
            x0=x0,
            y0=y0,
            theta0=theta0,
            metricas_base=metricas_base,
            pesos=pesos,
            x_scaler_model=x_scaler_model,
            y_scaler_model=y_scaler_model,
            l_input=l_input,
            imprimir=False,
        )

    resultado = differential_evolution(
        objetivo,
        bounds=PARAMETER_BOUNDS,
        strategy="best1bin",
        maxiter=maxiter,
        popsize=popsize,
        tol=1e-6,
        mutation=(0.5, 1.0),
        recombination=0.7,
        seed=SEED,
        polish=True,
        workers=1,
        updating="immediate",
        disp=True,
    )

    return resultado, metricas_base, resultado_base


In [ ]:

# ============================================================
# 12. Visualização dos resultados
# ============================================================

def comparar_resultados(
    xr,
    yr,
    resultado_base,
    resultado_otimizado,
    metricas_base,
    metricas_otimizado,
):
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))

    axs[0, 0].plot(xr, yr, "k--", label="Referência")
    axs[0, 0].plot(
        resultado_base["x_hat"],
        resultado_base["y_hat"],
        label="Controlador base"
    )
    axs[0, 0].plot(
        resultado_otimizado["x_hat"],
        resultado_otimizado["y_hat"],
        label="Controlador otimizado"
    )
    axs[0, 0].set_title("Trajetória no plano XY")
    axs[0, 0].set_xlabel("x [m]")
    axs[0, 0].set_ylabel("y [m]")
    axs[0, 0].axis("equal")
    axs[0, 0].grid(True)
    axs[0, 0].legend()

    axs[0, 1].plot(resultado_base["theta_hat"], label="Base")
    axs[0, 1].plot(resultado_otimizado["theta_hat"], label="Otimizado")
    axs[0, 1].set_title("Orientação estimada")
    axs[0, 1].set_xlabel("Amostra")
    axs[0, 1].set_ylabel("theta [rad]")
    axs[0, 1].grid(True)
    axs[0, 1].legend()

    axs[1, 0].plot(resultado_base["pwm_d"], label="PWM D base")
    axs[1, 0].plot(resultado_base["pwm_e"], label="PWM E base")
    axs[1, 0].plot(resultado_otimizado["pwm_d"], "--", label="PWM D otimizado")
    axs[1, 0].plot(resultado_otimizado["pwm_e"], "--", label="PWM E otimizado")
    axs[1, 0].set_title("Sinais PWM")
    axs[1, 0].set_xlabel("Amostra")
    axs[1, 0].grid(True)
    axs[1, 0].legend()

    nomes = ["Er", "Etraj", "Eu", "Ea"]
    x = np.arange(4)
    largura = 0.35

    axs[1, 1].bar(x - largura/2, metricas_base, largura, label="Base")
    axs[1, 1].bar(x + largura/2, metricas_otimizado, largura, label="Otimizado")
    axs[1, 1].set_xticks(x)
    axs[1, 1].set_xticklabels(nomes)
    axs[1, 1].set_title("Comparação dos termos da função de custo")
    axs[1, 1].grid(axis="y")
    axs[1, 1].legend()

    plt.tight_layout()
    plt.show()


def imprimir_resumo(resultado_otimizacao, metricas_base, metricas_otimizado):
    print("========== RESULTADO DA OTIMIZAÇÃO ==========")
    print("Sucesso:", resultado_otimizacao.success)
    print("Mensagem:", resultado_otimizacao.message)
    print("Custo final:", resultado_otimizacao.fun)
    print("Parâmetros ótimos:")
    print(resultado_otimizacao.x)

    print("\nMétricas base:")
    for nome, valor in zip(["Er", "Etraj", "Eu", "Ea"], metricas_base):
        print(f"{nome:6s}: {valor:.8f}")

    print("\nMétricas otimizadas:")
    for nome, valor in zip(["Er", "Etraj", "Eu", "Ea"], metricas_otimizado):
        print(f"{nome:6s}: {valor:.8f}")


In [ ]:

# ============================================================
# 13. Execução principal
# ============================================================

# Descomente e ajuste os nomes dos arquivos antes de executar.

# modelo_pirnn, x_scaler, y_scaler = carregar_modelos()
# referencia_df, xr, yr, theta_ref = carregar_referencia()

# Parâmetros de João ou parâmetros iniciais do controlador
# parametros_base = np.array([
#     0.15129693,
#     0.12799619,
#     0.00358131,
#     0.18509992,
#     0.97523155
# ])

# Condição inicial
# x0 = xr[0]
# y0 = yr[0]
# theta0 = theta_ref[0]

# resultado_otimizacao, metricas_base, resultado_base = otimizar_controlador(
#     modelo=modelo_pirnn,
#     xr=xr,
#     yr=yr,
#     theta_ref=theta_ref,
#     parametros_base=parametros_base,
#     x0=x0,
#     y0=y0,
#     theta0=theta0,
#     x_scaler_model=x_scaler,
#     y_scaler_model=y_scaler,
#     l_input=3.0,
#     pesos=WEIGHTS,
#     maxiter=50,
#     popsize=8,
# )

# resultado_otimizado = simular_malha_fechada_pirnn(
#     parametros_fuzzy=resultado_otimizacao.x,
#     modelo=modelo_pirnn,
#     xr=xr,
#     yr=yr,
#     theta_ref=theta_ref,
#     x0=x0,
#     y0=y0,
#     theta0=theta0,
#     x_scaler_model=x_scaler,
#     y_scaler_model=y_scaler,
#     l_input=3.0,
# )

# metricas_otimizado = calcular_metricas(
#     xr,
#     yr,
#     resultado_otimizado["x_hat"],
#     resultado_otimizado["y_hat"],
#     resultado_otimizado["phi_d"],
#     resultado_otimizado["phi_e"],
# )

# imprimir_resumo(
#     resultado_otimizacao,
#     metricas_base,
#     metricas_otimizado
# )

# comparar_resultados(
#     xr,
#     yr,
#     resultado_base,
#     resultado_otimizado,
#     metricas_base,
#     metricas_otimizado
# )



## Ajustes obrigatórios antes da execução

1. **Modelo e scalers:** informe os nomes corretos em `MODEL_PATH`, `X_SCALER_PATH` e `Y_SCALER_PATH`.
2. **Entradas da PIRNN:** confirme `PIRNN_INPUT_COLUMNS`.
3. **Saídas da PIRNN:** confirme se são `dX`, `dY`, `dTheta` ou `X`, `Y`, `Theta`.
4. **Referencial dos incrementos:** confirme se `dX` e `dY` estão no referencial global ou no referencial do robô.
5. **Conversão PWM:** substitua `rodas_para_pwm()` pela relação real utilizada no treinamento.
6. **Controlador:** o código mantém a estrutura fuzzy de João e otimiza os cinco parâmetros.
7. **Pesos:** os valores em `WEIGHTS` são iniciais. Eles podem ser modificados após análise de sensibilidade.
